# Tool 4: Harmonize the scored-event labels

This tool lets you **visualize** the scored-event configurations present in a heterogeneous database of recordings (events
annotated during sleep scoring and exported by Profusion/Compumedics: apnea, hypopnea, arousals, limb movements, PLM,
SpO2 desaturation…) and **harmonize** their raw labels to a single canonical vocabulary.

- **Inputs**: a study folder holding the recordings (`.edf`, BrainVision `.vhdr` or Curry `.cdt`), each with its scored-event export
  next to it (see Details).
- **Outputs**: `config_param/event_remap.json` in the study folder, a flat Python dict `{raw_label: canonical_label}`
  (`null` for the labels you choose to ignore) that the later tools use (sleep macrostructure, epoch flagging), its
  name noted in `config_param/file_names.json` with the **event files of the study** (which kinds of companion it
  has, so that the later tools look only for those); with the optional check, `config_param/event_source_mismatch.tsv`.
- **Sections**: 1. Study folder (scan the events; optionally, check that the text/CSV export and the `*.edf.XML`
  agree), 2. Event configurations found, 3. Harmonize the labels, 4. Preview & save the JSON, 5. Verify.

<details style="border:1px solid #bbb; border-radius:4px; background:#f7f7f7; padding:8px 12px; margin:10px 0;"><summary style="cursor:pointer; font-weight:bold;">Details <span style="font-size:85%; font-weight:normal; color:#666;">(click to open)</span></summary>

The events are read, in priority order, from the `*_ScoredEvents_Export.txt` text export (Compumedics/Curry French
export), then the `*_event_xml.csv`, then the `<ScoredEvents>` of the `*.edf.XML` (Profusion `CMPStudyConfig`).

</details>

In [ ]:
# Imports
try:
    import os
    import re
    import json
    import datetime
    import traceback
    import unicodedata
    import xml.etree.ElementTree as ET
    from pathlib import Path
    from collections import Counter, OrderedDict
    import pandas as pd
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML, clear_output
except ImportError as e:
    print('❌ Import error, a package is missing:', e)

# the look shared by the Voila pages (tools/ui_lib.py), found from the repo root, from tools/ or from a folder beside it
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'), os.path.join(os.path.dirname(os.getcwd()), 'tools')]:
    if os.path.isfile(os.path.join(_p, 'ui_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import ui_lib as UI
import study_files_lib as SF   # the choice of the companion suffixes
import recording_lib as RL     # one way to read a recording (EDF, BrainVision, Curry)


# ---- generic helpers (shared idioms with 2_select&remap_channels_edf) ----
def _load_json_lenient(path):
    """Load a JSON file, tolerating a single trailing comma before a closing } or ]
    (a common hand-edit mistake): strict parse first, repair only on failure."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r",(\s*[}\]])", r"\1", text))


def print_in_scrollable_box(text, height=300, font_size="12px"):
    display(HTML(f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; '
                 f'padding:10px; font-size:{font_size};">{text}</pre>'))


# ---- canonical event vocabulary (editable suggestions only) ----
# Compumedics/Profusion raw labels -> harmonized snake_case canonical labels.
# Arousal subtypes are kept (clinically meaningful); laterality is dropped for limbs.
DEFAULT_EVENT_MAPPING = {
    "obstructive apnea":   "apnea_obstructive",
    "central apnea":       "apnea_central",
    "mixed apnea":         "apnea_mixed",
    "hypopnea":            "hypopnea",
    "spo2 desaturation":   "spo2_desaturation",
    "spo2 artifact":       "spo2_artifact",
    "arousal (aro res)":   "arousal_respiratory",
    "arousal (aro spont)": "arousal_spontaneous",
    "arousal (aro plm)":   "arousal_limb",
    "arousal (aro limb)":  "arousal_limb",
    "arousal ()":          "arousal",
    "arousal":             "arousal",
    "limb movement (left)":  "limb_movement",
    "limb movement (right)": "limb_movement",
    "limb movement":         "limb_movement",
    "plm (left)":  "plm",
    "plm (right)": "plm",
    "plm":         "plm",
    "snore":       "snore",
    "snoring":     "snore",
}
# ---- French text-export labels (*_ScoredEvents_Export.txt) ----
# The French export writes variable label text ("Micro-éveil 2 ARO RES"), so match on
# informative substrings (accent-insensitive) rather than exact keys.
FRENCH_EVENT_RULES = [
    ("aro spont",         "arousal_spontaneous"),
    ("aro res",           "arousal_respiratory"),
    ("aro plm",           "arousal_limb"),
    ("aro limb",          "arousal_limb"),
    ("apnee obstructive", "apnea_obstructive"),
    ("apnee centrale",    "apnea_central"),
    ("apnee mixte",       "apnea_mixed"),
    ("hypopnee",          "hypopnea"),
    ("desaturation",      "spo2_desaturation"),
    ("artefact spo2",     "spo2_artifact"),
    ("ronflement",        "snore"),
    ("plm",               "plm"),
]
CANONICAL_VOCAB = sorted(set(DEFAULT_EVENT_MAPPING.values())
                         | {canon for _, canon in FRENCH_EVENT_RULES})


def _strip_accents(text):
    """Lower-case and drop accents so French labels match regardless of accentuation
    (é→e, É→e). Used for the substring rules above."""
    return "".join(c for c in unicodedata.normalize("NFKD", text.lower())
                   if not unicodedata.combining(c))


def suggest_canonical(raw):
    """Suggest a canonical label for a raw event name (empty string if unknown).
    English labels (Compumedics CSV/XML) match an exact dict; French labels (the
    *_ScoredEvents_Export.txt export) match accent-insensitive substrings."""
    key = raw.strip().lower()
    if key in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[key]
    # tolerate a trailing laterality marker, e.g. "Limb Movement (Left)"
    stripped = re.sub(r"\s*\((left|right)\)\s*$", "", key).strip()
    if stripped in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[stripped]
    # French text export: informative-substring match (accent-insensitive)
    norm = _strip_accents(raw)
    for sub, canon in FRENCH_EVENT_RULES:
        if sub in norm:
            return canon
    return ""


# ---- event companion loading: TXT first, then CSV, then XML (<ScoredEvents>) fallback ----
# one copy, in edf_io_lib (the Curry generator that rewrote these lines is gone: phase 6 of tools/plan_brainvision.md)
from edf_io_lib import _TXT_DUR_RE, event_companion_paths


def load_events_from_csv(csv_path):
    """Parse a Compumedics *_event_xml.csv -> list of (name, start, duration)."""
    df = pd.read_csv(csv_path)
    for col in ("Name", "Start", "Duration"):
        if col not in df.columns:
            raise ValueError(f"missing column '{col}' in {Path(csv_path).name}")
    events = []
    for _, row in df.iterrows():
        events.append((str(row["Name"]).strip(), float(row["Start"]), float(row["Duration"])))
    return events


def load_events_from_xml(xml_path):
    """Parse the <ScoredEvents> of a Profusion CMPStudyConfig .edf.XML
    -> list of (name, start, duration). <Input> is ignored (absent from the CSV)."""
    root = ET.parse(xml_path).getroot()
    events = []
    for se in root.iter("ScoredEvent"):
        name_el = se.find("Name")
        if name_el is None or name_el.text is None:
            continue
        start_el = se.find("Start")
        dur_el = se.find("Duration")
        start = float(start_el.text) if (start_el is not None and start_el.text) else float("nan")
        dur = float(dur_el.text) if (dur_el is not None and dur_el.text) else float("nan")
        events.append((name_el.text.strip(), start, dur))
    return events


def load_events_from_txt(txt_path, rec_start=None):
    """Parse a Compumedics/Curry French text export (*_ScoredEvents_Export.txt)
    -> list of (name, start, duration). Comma-separated, no header, columns:
        HH:MM:SS , epoch# , stage_FR , event_label_FR , M:SS[.s] , - , - , position
    Encoding varies (UTF-16 with BOM on some exports, UTF-8/ANSI on others) -> the BOM is
    sniffed. Clock times are converted to seconds-from-recording-start (with midnight
    rollover) when rec_start is given; without it (harmonization only needs the names)
    Start and Duration are returned as NaN, so no EDF-header read is required for the scan."""
    raw = open(txt_path, "rb").read()
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        text = raw.decode("utf-16")
    else:
        text = raw.decode("utf-8", errors="replace")
    rec_date = rec_start.date() if rec_start is not None else None
    events = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 5:
            continue
        name = parts[3]
        if rec_start is None:
            events.append((name, float("nan"), float("nan")))
            continue
        # clock time -> seconds from recording start (hours may be single-digit)
        try:
            hh, mm, ss = parts[0].split(":")
            clock_t = datetime.time(int(hh), int(mm), int(ss))
        except (ValueError, TypeError):
            events.append((name, float("nan"), float("nan")))
            continue
        event_dt = datetime.datetime.combine(rec_date, clock_t)
        # events recorded after midnight fall on the next calendar day
        if (rec_start - event_dt).total_seconds() > 3600:
            event_dt += datetime.timedelta(days=1)
        start_sec = (event_dt - rec_start).total_seconds()
        m = _TXT_DUR_RE.match(parts[4])
        dur_sec = int(m.group(1)) * 60 + float(m.group(2)) if m else 0.0
        events.append((name, start_sec, dur_sec))
    return events


def load_events(edf_path, txt_suffix="_ScoredEvents_Export.txt", csv_suffix="_event_xml.csv", xml=True):
    """TXT-first, then CSV, then XML-fallback event loader.
    Returns (events_list, source) with source in {'txt', 'csv', 'xml'} or (None, None) when
    no companion is usable. events_list = list of (name, start, duration). For the TXT source
    only the names are needed here (harmonization), so Start/Duration are left NaN (no EDF
    header read); the 1bis check reads them with the recording-start datetime. A kind the study
    does not have (suffix None, xml=False) is not looked for."""
    txt, csv, xml = event_companion_paths(edf_path, txt_suffix, csv_suffix, xml)
    if txt is not None:
        try:
            return load_events_from_txt(txt), "txt"
        except Exception:
            pass  # fall through to the CSV
    if csv is not None:
        try:
            return load_events_from_csv(csv), "csv"
        except Exception:
            pass  # fall through to the XML
    if xml is not None:
        try:
            return load_events_from_xml(xml), "xml"
        except Exception:
            pass
    return None, None


def load_existing_mapping():
    """Read the existing mapping (lenient): the file of the 'Inputs found' list, found in the study folder or chosen
    by hand (another cohort's, to start from). {} if absent/unreadable."""
    p = fc_mapping.selected
    if p and Path(p).exists():
        try:
            return _load_json_lenient(p)
        except Exception:
            return {}
    return {}


def _canon_events(events):
    """List of (canonical_label, floor(start_seconds)) for cross-source comparison.
    Names are normalized to the canonical vocabulary (suggest_canonical, raw-lowercase fallback)
    so English (CSV/XML) and French (.txt) labels compare equal; start times are floored to whole
    seconds because the .txt export truncates clock times to the second."""
    return [(suggest_canonical(n) or n.strip().lower(), int(s)) for n, s, _d in events]


def _match_events(primary, xml, tol):
    """Greedy match of primary events to XML events within +/- tol whole seconds.
    primary / xml: lists of (canonical_label, floor_second) from _canon_events.
    Returns (n_matched, cooccur_pairs, only_in_primary, only_in_xml):
      - pass 1 pairs events with the SAME canonical label within tol (nearest second wins);
      - pass 2 pairs any leftover events within tol -> a co-occurring pair with DIFFERENT labels,
        recorded in cooccur_pairs as a {(primary_label, xml_label): count} Counter. Inspect those
        pairs to lift the ambiguity between "same event, unharmonized name" (e.g. a cross-language
        pair worth mapping together) and "two genuinely distinct events that happen to fall within
        tol seconds" (unrelated labels);
      - anything still unpaired is only-in-one-source.
    tol=0 reduces to strict same-second matching. Greedy O(n*m) — fine for a few hundred events."""
    P, X = list(primary), list(xml)
    used_x = [False] * len(X)
    matched_p = [False] * len(P)
    n_matched = 0
    cooccur = Counter()

    def _nearest(ps, pl, same_label):
        best_j, best_d = -1, tol + 1
        for j, (xl, xs) in enumerate(X):
            if used_x[j] or (same_label and xl != pl):
                continue
            d = abs(ps - xs)
            if d <= tol and d < best_d:
                best_j, best_d = j, d
        return best_j

    for i, (pl, ps) in enumerate(P):                 # pass 1: same label within tol
        j = _nearest(ps, pl, True)
        if j >= 0:
            used_x[j] = matched_p[i] = True
            n_matched += 1
    for i, (pl, ps) in enumerate(P):                 # pass 2: any label within tol -> co-occurrence
        if matched_p[i]:
            continue
        j = _nearest(ps, pl, False)
        if j >= 0:
            used_x[j] = matched_p[i] = True
            cooccur[(pl, X[j][0])] += 1
    only_p = Counter(pl for i, (pl, _s) in enumerate(P) if not matched_p[i])
    only_x = Counter(xl for j, (xl, _s) in enumerate(X) if not used_x[j])
    return n_matched, cooccur, only_p, only_x


# ---- shared state filled by the scan ----
STATE = {}


# ========================= Section banners =========================
section1 = widgets.HTML(UI.section_html(1, 'Study folder', """
<p>Pick the study folder (recordings: <code>.edf</code>, BrainVision <code>.vhdr</code> or Curry <code>.cdt</code>). Each recording is expected to
have a Compumedics/Profusion event companion next to it. Three sources are supported, in priority order:
<br>&#x2022; the <b>text export</b> <code>*_ScoredEvents_Export.txt</code> (read first; suffix in
the <b>TXT suffix</b> field: default <code>_ScoredEvents_Export.txt</code>);
<br>&#x2022; then the <b>CSV</b> <code>*_event_xml.csv</code> (suffix in the <b>CSV suffix</b> field);
<br>&#x2022; then, as a fallback, the <code>&lt;ScoredEvents&gt;</code> of the <code>*.edf.XML</code>.
<br>&#x2022; Selecting the folder auto-detects both suffixes, finds the existing mapping (<b>Inputs found</b>;
<i>Choose another…</i> starts from another file, e.g. another cohort's) and refreshes the info lines below. The
<b>Advanced</b> box holds the name of the file Section 4 saves.
<br>&#x2022; <b>Event files of this study</b>: one box per kind of file (the three above, and the Profusion
<code>*_Summary_Export.txt</code> that tool 5 reads for lights-off / lights-on), pre-ticked from the files found
beside the recordings, or as saved before. Untick what the study does not have: tools 5, 7 and 0 then no longer look
for it nor warn about it. The boxes are saved with the mapping (Section 4), or alone with <b>Save the declaration</b>
(a study without scored events: untick every box, then save).
<br>&#x2022; Click <b>Scan</b> to read the events and list the configurations.
<br>&#x2022; "Skip labels already mapped" hides labels already present in an existing
<code>event_remap.json</code> (incremental harmonization when you add a new cohort).</p>
"""))

section1bis = widgets.HTML(UI.subsection_html(None, 'Optional: check text/CSV vs XML consistency', """
<p>For every file having <b>both</b> a primary source (the <code>.txt</code> text export if present,
else the <code>*_event_xml.csv</code>) <b>and</b> the <code>&lt;ScoredEvents&gt;</code> of the
<code>*.edf.XML</code>, checks that the two describe the same events. Labels are normalized to the
<b>canonical vocabulary</b> (so English XML and French <code>.txt</code> compare equal) and events are
matched by <b>type + start time within ±(Match&nbsp;tolerance) seconds</b> (default 1&nbsp;s: the
<code>.txt</code> truncates clock times to the second and the two exports can round a start
differently; set 0 for strict same-second matching). Events paired within the tolerance but carrying
<b>different labels</b> are listed in <code>cooccur_label_pairs</code> as candidate same-events whose
names are not yet harmonized (e.g. a cross-language pair): <b>inspect those pairs to decide whether
they are truly one event or two distinct events that merely fall within the tolerance</b>. Events with
no counterpart show up as only-in-one-source (e.g. an export that omits snoring). Writes
<code>config_param/event_source_mismatch.tsv</code>. Opt-in because it reads both files per recording.</p>
"""))

section2 = widgets.HTML(UI.section_html(2, 'Event configurations found', """
<p>Files are grouped by their set of <b>unique</b> event labels. Two files with the same unique
labels share one configuration even if their event counts differ.</p>
"""))

section3 = widgets.HTML(UI.section_html(3, 'Harmonize the labels', """
<p>For each unique raw label, choose a harmonized (canonical) label. Defaults are pre-filled
when recognized; you can edit them freely.
<br>&#x2022; Tick <b>ignore</b> to drop a label (stored as <code>null</code>, excluded downstream).
<br>&#x2022; Click <b>Validate mapping &amp; ignores</b> to check your choices and unlock section 4.
<br>&#x2022; Then go to section 4 to preview &amp; save.</p>
"""))

section4 = widgets.HTML(UI.section_html(4, 'Preview &amp; save the JSON', """
<p>Builds a flat <code>{raw_label: canonical_label}</code> mapping and <b>merges</b> it into any
existing <code>config_param/event_remap.json</code> (labels mapped this session replace their old
value, all others are kept).</p>
"""))

section5 = widgets.HTML(UI.section_html(5, 'Verify', """
<p>Applies the saved mapping to every configuration and reports the resulting harmonized labels.
The verdict passes when no raw label is left unmapped (ignored labels count as handled).</p>
"""))

# ========================= Section 1 widgets =========================
chooser = FileChooser(os.getcwd())
chooser.title = UI.picker_title('Study folder')
chooser.show_only_dirs = True

# The input file, found in the study folder (the 'Inputs found' list): its picker shows under its line, with
# "Choose another…", for a file chosen by hand. The scan and the merge read its selection.
fc_mapping = FileChooser(filter_pattern='*.json')
fc_mapping.title = UI.picker_title('Existing mapping', '(an <code>event_remap.json</code>: the labels it maps are kept)')
mapping_row, locate_mapping = UI.input_row(fc_mapping, 'Existing mapping', 'optional', 'a new mapping is started',
                                           check=SF.check_event_remap)
inputs_list, show_inputs = UI.inputs_list([mapping_row])
# The file Section 4 saves in config_param/: a study keeps its name (study_files_lib.config_file_name)
fname_text = widgets.Text(value=SF.EVENT_REMAP_DEFAULT, description="File name:",
                          style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
fname_help = widgets.HTML(UI.help_html('The JSON file Section 4 saves in <code>config_param/</code>. The later tools '
                                       'find it under this name (noted in <code>config_param/file_names.json</code>).'))
advanced, mark_advanced = UI.advanced_box([fname_text, fname_help])
_proposed_name = {'name': SF.EVENT_REMAP_DEFAULT}

skip_existing = widgets.Checkbox(value=True, description="Skip labels already mapped",
                                 style={"description_width": "initial"})
existing_info = widgets.HTML(value="")
txt_suffix = widgets.Text(value="_ScoredEvents_Export.txt", description="TXT suffix:",
                          style=UI.LABEL_STYLE,
                          layout=widgets.Layout(width=UI.FIELD_WIDTH))
txt_suffix_info = widgets.HTML(value="")
csv_suffix = widgets.Text(value="_event_xml.csv", description="CSV suffix:",
                          style=UI.LABEL_STYLE,
                          layout=widgets.Layout(width=UI.FIELD_WIDTH))
csv_suffix_info = widgets.HTML(value="")
# a menu of the detected suffixes in front of each suffix field (the field stays the value the scan reads)
txt_menu, fill_txt_menu = UI.suffix_menu(txt_suffix, txt_suffix_info)
csv_menu, fill_csv_menu = UI.suffix_menu(csv_suffix, csv_suffix_info)
summary_suffix = widgets.Text(value="_Summary_Export.txt", description="Summary suffix:",
                              style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
summary_suffix_info = widgets.HTML(value="")
summary_menu, fill_summary_menu = UI.suffix_menu(summary_suffix, summary_suffix_info)
# The event files of the study (study_files_lib.declared_event_files): one box per kind, pre-ticked from the files
# found beside the recordings or from the declaration saved. A suffix field shows only for a kind ticked.
cb_files = {kind: widgets.Checkbox(value=True, description=label, indent=False,
                                   layout=widgets.Layout(width='max-content', margin='0 18px 0 0'))
            for kind, label in SF.EVENT_FILE_KINDS.items()}
files_row = widgets.HBox([widgets.HTML('<b>Event files of this study:</b>',
                                       layout=widgets.Layout(width=UI.LABEL_WIDTH))] + list(cb_files.values()))
files_info = widgets.HTML(value="")
suffix_boxes = {'txt': widgets.VBox([txt_menu, txt_suffix, txt_suffix_info]),
                'csv': widgets.VBox([csv_menu, csv_suffix, csv_suffix_info]),
                'summary': widgets.VBox([summary_menu, summary_suffix, summary_suffix_info])}
save_files_button = UI.button("Save the declaration", "save", icon="save",
                              tooltip="Save the boxes of 'Event files of this study' without a mapping")
out_files = widgets.HTML(value="")
run_scan_button = UI.button("Scan", "read")
scan_progress, scan_progress_row = UI.progress_bar("Files:")
out_scan = widgets.Output()

# Section 1bis
run_check_button = UI.button("Check text/CSV vs XML", "run", icon="check")
tol_seconds = widgets.BoundedIntText(value=1, min=0, max=10, description="Match tolerance (s):",
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="170px"))
out_check = widgets.Output()

# Section 2 / 3 / 4 / 5 output zones
out_configs = widgets.Output()
out_harmonize = widgets.Output()
preview_save_button = UI.button("Preview & Save", "save", disabled=True)
out_save = widgets.Output()
validate_button = UI.button("Validate mapping & ignores", "save", icon="check")
out_validate = widgets.Output()
verify_scope = widgets.Dropdown(options=["All configurations", "Only configs with unmapped labels"],
                                value="All configurations", description="Show:",
                                style={"description_width": "initial"}, layout=widgets.Layout(width="380px"))
verify_button = UI.button("Verify", "run")
out_verify = widgets.Output()


# ========================= Callbacks =========================
def _update_advanced_mark(change=None):
    """'⚠️ default changed' in the title of the Advanced box when the file name differs from the one proposed."""
    mark_advanced(['file name'] if fname_text.value.strip() != _proposed_name['name'] else [])


def _find_mapping(folder):
    """The existing mapping of the study (its line in the list) and the name Section 4 saves under. Non-fatal."""
    try:
        names, names_warn = SF.load_names(folder)
        _proposed_name['name'] = SF.config_file_name(folder, names, 'event_remap', SF.EVENT_REMAP_DEFAULT)
        fname_text.value = _proposed_name['name']
        _update_advanced_mark()
        locate_mapping(folder, *SF.find_input(folder, names, 'event_remap'))
        return f"<br>{UI.msg('warn', names_warn)}" if names_warn else ''
    except Exception as e:
        return f"<br>{UI.msg('error', f'Error looking for the existing mapping: {e}')}"


def _show_suffix_boxes(change=None):
    """A suffix field shows only for a kind of file the study has (its box ticked)."""
    for kind, box in suffix_boxes.items():
        box.layout.display = '' if cb_files[kind].value else 'none'


def declaration():
    """The boxes of 'Event files of this study' as study_files_lib.declared_event_files returns a declaration:
    {'txt': suffix or None, 'csv': suffix or None, 'xml': True / False, 'summary': suffix or None}."""
    def suffix(kind, field):
        return (field.value.strip() or None) if cb_files[kind].value else None
    return {'txt': suffix('txt', txt_suffix), 'csv': suffix('csv', csv_suffix), 'xml': bool(cb_files['xml'].value),
            'summary': suffix('summary', summary_suffix)}


def save_declaration(folder):
    """Record the boxes in config_param/file_names.json (tools 5, 7 and 0 read them). Returns the line to show."""
    decl = declaration()
    warn = SF.write_names(folder, '4', {SF.EVENT_FILES_KEY: decl})
    if warn:
        return UI.msg('warn', warn)
    kinds = [SF.EVENT_FILE_KINDS[k] for k, v in decl.items() if v]
    return UI.msg('ok', 'Event files of this study saved in <code>config_param/file_names.json</code>: '
                  + (', '.join(kinds) if kinds else 'none (no scored events)') + '.')


def on_save_files(_=None):
    try:
        if not getattr(chooser, "selected_path", None):
            out_files.value = UI.msg('warn', 'Please select your study folder first.')
            return
        out_files.value = save_declaration(Path(chooser.selected_path))
    except Exception as e:
        out_files.value = UI.msg('error', f'Declaration not saved: {type(e).__name__}: {e}')


def _update_info(*_):
    """Refresh the info line when the folder changes (no scan): the suffixes and the boxes of 'Event files of this
    study', from the declaration saved, else from the files found."""
    fill_txt_menu({}, [])   # emptied until the detection below
    fill_csv_menu({}, [])
    fill_summary_menu({}, [])
    out_files.value = ""
    if not getattr(chooser, "selected_path", None):
        show_inputs(False)
        locate_mapping(None)
        existing_info.value = ""
        csv_suffix_info.value = ""
        files_info.value = ""
        return
    show_inputs(True)
    mapping_note = _find_mapping(Path(chooser.selected_path))
    try:
        folder = Path(chooser.selected_path)
        edfs = SF.list_recordings(folder)
        if not edfs:
            existing_info.value = UI.msg('warn', 'No recording (.edf, .vhdr, .cdt) found in selected folder.')
            csv_suffix_info.value = ""
            txt_suffix_info.value = ""
            files_info.value = ""
            return
        existing = load_existing_mapping()
        msg = f"<small>{len(edfs)} recording(s) found. "
        msg += (f"{len(existing)} label(s) already mapped."
                if existing else "No existing mapping yet.")
        conflicts = SF.recording_conflicts_text(folder)
        existing_info.value = (msg + "</small>" + mapping_note
                               + (f"<br>{UI.msg('warn', conflicts)}" if conflicts else ""))
        stems = [f.stem for f in edfs]
        names, _ = SF.load_names(folder)
        declared = SF.declared_event_files(names)      # None: nothing declared, the boxes follow the files found

        def detect(kind, field, fill_menu, info, owners, absent_msg):
            """One suffix field: the suffix saved in the declaration, else the most frequent found (shortest on
            ties: events have no 'more specific remapped' variant, unlike hypnograms). Returns the number of
            recordings having a file of that kind."""
            if owners:
                field.value, _ = SF.select_event_suffix({s: len(r) for s, r in owners.items()})
            if declared and declared[kind]:
                field.value = declared[kind]
            if owners:
                fill_menu(owners, stems)
            else:
                info.value = f'<small style="color:{UI.WARN_COLOR};">{absent_msg}</small>'
            return len({r for rs in owners.values() for r in rs})

        all_txt = [f for f in folder.rglob("*") if f.suffix.lower() == ".txt"]
        found = {}
        # --- Event TXT-export suffix auto-detection (Compumedics/Curry *_ScoredEvents_Export.txt) ---
        # Scan .txt files whose name contains 'event' so hypnogram .txt files are not counted.
        found['txt'] = detect('txt', txt_suffix, fill_txt_menu, txt_suffix_info,
                              SF.companion_suffix_owners(edfs, [f for f in all_txt if "event" in f.name.lower()]),
                              'No event .txt detected next to the recordings — set the suffix manually or rely on CSV/XML.')
        # --- Event-CSV suffix auto-detection ---
        all_csv = [f for f in folder.rglob("*") if f.suffix.lower() == ".csv"]
        found['csv'] = detect('csv', csv_suffix, fill_csv_menu, csv_suffix_info, SF.companion_suffix_owners(edfs, all_csv),
                              'No event CSV detected next to the recordings — set the suffix manually or rely on the '
                              'XML fallback.')
        found['xml'] = sum(1 for e in edfs if event_companion_paths(e, None, None)[2] is not None)
        # --- Profusion summary export (*_Summary_Export.txt): tool 5's lights-off / lights-on ---
        found['summary'] = detect('summary', summary_suffix, fill_summary_menu, summary_suffix_info,
                                  SF.companion_suffix_owners(edfs, [f for f in all_txt if "summary" in f.name.lower()]),
                                  'No summary export detected next to the recordings — set the suffix manually.')
        # --- the boxes of 'Event files of this study' ---
        for kind, cb in cb_files.items():
            cb.value = bool(declared[kind]) if declared else found[kind] > 0
        _show_suffix_boxes()
        counts = ', '.join(f'{SF.EVENT_FILE_KINDS[k]} {n}/{len(edfs)}' for k, n in found.items())
        if declared:
            note = UI.msg('muted', 'As declared for this study (<code>config_param/file_names.json</code>). Found beside '
                                   f'the recordings: {counts}.')
            missed = [SF.EVENT_FILE_KINDS[k] for k, n in found.items() if n and not declared[k]]
            if missed:
                note += '<br>' + UI.msg('warn', f'Declared absent, yet found beside some recordings: {", ".join(missed)}. '
                                                'Tick it and save the declaration for the tools to read it.')
        else:
            note = UI.msg('muted', f'Pre-ticked from the files found beside the recordings: {counts}. Saved with the '
                                   'mapping (Section 4) or with <b>Save the declaration</b>.')
        files_info.value = UI.text_block(f'<small>{note}</small>')
    except Exception as e:
        existing_info.value = UI.msg('error', f'{type(e).__name__}: {e}')
        csv_suffix_info.value = ""


def run_scan(_=None):
    for z in (out_scan, out_configs, out_harmonize, out_validate, out_save, out_verify):
        z.clear_output()
    with out_scan:
        if not getattr(chooser, "selected_path", None):
            UI.log("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = SF.list_recordings(folder)
        if not edfs:
            UI.log("⚠️ No recording (.edf, .vhdr, .cdt) found in the selected folder.")
            return
        decl = declaration()     # the kinds unticked in 'Event files of this study' are not looked for
        if not (decl['txt'] or decl['csv'] or decl['xml']):
            print("ℹ️ No scored-event file ticked in 'Event files of this study': nothing to scan. For a study without "
                  "scored events, click 'Save the declaration' so that tools 5, 7 and 0 know.")
            return
        configs = {}          # frozenset(labels) -> [file_id]
        label_files = {}      # raw label -> set(file_id)
        label_counts = {}     # raw label -> total occurrences
        source_by_file = {}   # file_id -> 'txt'/'csv'/'xml'
        failed = []           # (file_id, reason)
        n_with = 0
        scan_progress.max = max(1, len(edfs))
        scan_progress.value = 0
        scan_progress.bar_style = 'info'
        for edf in edfs:
            scan_progress.value += 1
            fid = edf.stem
            try:
                events, source = load_events(edf, decl['txt'], decl['csv'], decl['xml'])
            except Exception as e:
                failed.append((fid, f"{type(e).__name__}: {e}"))
                continue
            if events is None:
                failed.append((fid, "no readable event companion (.txt, .csv or .edf.XML)"))
                continue
            n_with += 1
            source_by_file[fid] = source
            names = [n for (n, _, _) in events]
            configs.setdefault(frozenset(names), []).append(fid)
            for nm in names:
                label_counts[nm] = label_counts.get(nm, 0) + 1
            for nm in set(names):
                label_files.setdefault(nm, set()).add(fid)
        UI.finish_progress(scan_progress, n_failed=len(failed))
        STATE.update(folder=folder, configs=configs, label_files=label_files,
                     label_counts=label_counts, source_by_file=source_by_file,
                     failed=failed, existing=load_existing_mapping())
        if failed:
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            pd.DataFrame(failed, columns=["file_id", "reason"]).to_csv(
                cfgdir / "failed_event_read.tsv", sep="\t", index=False)
        UI.log(f"✅ Scanned {len(edfs)} recording(s): {n_with} with events, {len(failed)} failed.")
        print(f"   {len(label_files)} unique raw label(s), {len(configs)} distinct event configuration(s).")
        n_txt = sum(1 for s in source_by_file.values() if s == "txt")
        n_csv = sum(1 for s in source_by_file.values() if s == "csv")
        n_xml = sum(1 for s in source_by_file.values() if s == "xml")
        print(f"   event source: {n_txt} from TXT, {n_csv} from CSV, {n_xml} from XML fallback.")
        if STATE["existing"]:
            print(f"   {len(STATE['existing'])} label(s) already in the existing mapping.")
        if failed:
            UI.log(f"   ⚠️ {len(failed)} file(s) without readable events "
                  f"— see config_param/failed_event_read.tsv")
    render_configs()
    render_harmonize()


def render_configs():
    with out_configs:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            return
        items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))

        # ---- per-config viewer: a dropdown selects one configuration to detail ----
        dd_options = [(f"Configuration {i}: {len(fids)} file(s), {len(labels)} unique label(s)", i - 1)
                      for i, (labels, fids) in enumerate(items, 1)]
        config_dd = widgets.Dropdown(options=dd_options, value=0, description="Show config:",
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="520px"))
        show_ids_btn = widgets.ToggleButton(value=False, description="Show file ids",
                                            icon="list-ul", layout=widgets.Layout(width="170px"))
        detail_out = widgets.Output()
        ids_out = widgets.Output()

        def _render_detail(*_):
            labels, fids = items[config_dd.value]
            labs = sorted(labels)
            with detail_out:
                clear_output()
                display(HTML(f"<b>{len(labs)} unique label(s)</b> in {len(fids)} file(s):<br>"
                             + "<br>".join(f"&#x2022; {l}" for l in labs)))
            show_ids_btn.description = "Hide file ids" if show_ids_btn.value else "Show file ids"
            with ids_out:
                clear_output()
                if show_ids_btn.value:
                    display(HTML('<pre style="overflow:auto; max-height:140px; border:1px solid #ccc; '
                                 'padding:6px; font-size:12px; margin-top:4px;">'
                                 + ", ".join(sorted(fids)) + "</pre>"))

        config_dd.observe(_render_detail, names="value")
        show_ids_btn.observe(_render_detail, names="value")
        display(widgets.VBox([config_dd, detail_out, show_ids_btn, ids_out]))
        _render_detail()

        # ---- global table of all unique raw labels (kept) ----
        lf, lc = STATE["label_files"], STATE["label_counts"]
        rows = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        df = pd.DataFrame([{"raw_label": l, "n_files": len(lf[l]), "n_occurrences": lc[l],
                            "suggested_canonical": suggest_canonical(l) or "(none)"} for l in rows])
        display(HTML("<h4>All unique raw labels</h4>"))
        display(HTML(df.to_html(index=False)))


def render_harmonize(*_):
    with out_harmonize:
        clear_output()
        out_validate.clear_output()
        preview_save_button.disabled = True   # require a fresh validation after any (re)build
        lf = STATE.get("label_files", {})
        if not lf:
            return
        existing = STATE.get("existing", {})
        labels = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        if skip_existing.value:
            labels = [l for l in labels if l not in existing]
        STATE["rows"] = {}
        row_widgets = []
        if not labels:
            display(HTML("<i>All raw labels are already mapped (uncheck “Skip labels "
                         "already mapped” to edit them).</i>"))
            return
        header = widgets.HBox([
            widgets.HTML("<b>raw label</b> (files)", layout=widgets.Layout(width="320px")),
            widgets.HTML("&rarr; <b>canonical label</b>", layout=widgets.Layout(width="240px")),
            widgets.HTML("<b>ignore</b>", layout=widgets.Layout(width="90px")),
        ])
        for raw in labels:
            if raw in existing:
                val = existing[raw]
                combo_val, ignore_val = ("", True) if val is None else (val, False)
            else:
                combo_val, ignore_val = suggest_canonical(raw), False
            lab = widgets.HTML(f"<code>{raw}</code> <small style='color:{UI.MUTED_COLOR}'>({len(lf[raw])})</small>",
                               layout=widgets.Layout(width="320px"))
            combo = widgets.Combobox(value=combo_val, options=CANONICAL_VOCAB, ensure_option=False,
                                     placeholder="canonical label", disabled=ignore_val,
                                     layout=widgets.Layout(width="240px"))
            ign = widgets.Checkbox(value=ignore_val, description="ignore", indent=False,
                                   layout=widgets.Layout(width="90px"))

            def _toggle(change, c=combo):
                c.disabled = change["new"]
            ign.observe(_toggle, names="value")
            combo.observe(_invalidate_save, names="value")
            ign.observe(_invalidate_save, names="value")
            STATE["rows"][raw] = (combo, ign)
            row_widgets.append(widgets.HBox([lab, combo, ign], layout=widgets.Layout(
                border="1px solid #e0e0e0", padding="4px", margin="0 0 2px 0",
                align_items="center", overflow="hidden")))
        box = widgets.VBox(row_widgets, layout=widgets.Layout(
            max_height="420px", overflow="auto", border="1px solid #ddd", padding="6px"))
        display(widgets.VBox([header, box]))


def _invalidate_save(*_):
    """Any change in section 3 invalidates a prior validation: the saved JSON must
    always reflect the latest selection, so re-lock section 4 until re-validated."""
    if not preview_save_button.disabled:
        preview_save_button.disabled = True
        out_save.clear_output()
        with out_validate:
            clear_output()
            display(HTML(f"<i style='color:{UI.WARN_COLOR}'>⚠️ Selection changed, click "
                         "<b>Validate mapping &amp; ignores</b> again before saving.</i>"))


def on_validate(_=None):
    with out_validate:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            UI.log("⚠️ Run the scan (section 1) first.")
            return
        mapped, ignored, empties = [], [], []
        for raw, (combo, ign) in rows.items():
            if ign.value:
                ignored.append(raw)
            elif combo.value.strip():
                mapped.append(raw)
            else:
                empties.append(raw)
        display(HTML(f"<b>Validation summary:</b> {len(mapped)} mapped, "
                     f"{len(ignored)} ignored, {len(empties)} left empty."))
        if empties:
            display(HTML(f"<span style='color:{UI.WARN_COLOR}'>⚠️ empty (will NOT be saved): "
                         + ", ".join(f"<code>{e}</code>" for e in empties)
                         + " — set a canonical label or tick “ignore”.</span>"))
        display(HTML(f"<span style='color:{UI.OK_COLOR}'>✅ Section 4 unlocked, you can now "
                     "<b>Preview &amp; Save</b> below.</span>"))
        preview_save_button.disabled = False


def on_preview_save(_=None):
    with out_save:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            UI.log("⚠️ Run the scan (section 1) first.")
            return
        try:
            session, empties = {}, []
            for raw, (combo, ign) in rows.items():
                if ign.value:
                    session[raw] = None
                else:
                    v = combo.value.strip()
                    if v:
                        session[raw] = v
                    else:
                        empties.append(raw)
            folder = STATE["folder"]
            existing = load_existing_mapping()   # reload fresh to merge
            merged = dict(existing)
            merged.update(session)
            merged = OrderedDict(sorted(merged.items(), key=lambda kv: kv[0].lower()))
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            out_json = cfgdir / (fname_text.value.strip() or SF.EVENT_REMAP_DEFAULT)
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(merged, f, indent=2, ensure_ascii=False)
            # the later tools find it under this name; from now on it is this tool's existing mapping too. The boxes of
            # 'Event files of this study' are saved with it: the later tools look only for the kinds ticked.
            names_warn = SF.write_names(folder, '4', {'event_remap': SF.path_to_record(folder, out_json),
                                                      SF.EVENT_FILES_KEY: declaration()})
            if names_warn:
                display(HTML(f"<p>{UI.msg('warn', names_warn)}</p>"))
            locate_mapping(folder, out_json, 'saved by this tool', out_json.parent)
            n_new = len(session)
            display(HTML(f"<p><b>✅ JSON saved here:</b> <code>{out_json}</code></p>"))
            display(HTML(f"<p style='color:{UI.MUTED_COLOR}'>{len(merged)} label(s) total "
                         f"(<b>{n_new}</b> added/updated this session); previous entries preserved.</p>"))
            if empties:
                display(HTML(f"<p style='color:{UI.WARN_COLOR}'>⚠️ left unmapped (not saved): "
                             + ", ".join(f"<code>{e}</code>" for e in empties)
                             + " — set a canonical label or tick “ignore”.</p>"))
            preview = json.dumps(merged, indent=2, ensure_ascii=False)
            print_in_scrollable_box(preview.replace("<", "&lt;").replace(">", "&gt;"), height=260)
            display(HTML("<br><b>To load this file later:</b>"))
            display(HTML("<p><code>with open(path, encoding='utf-8') as f: event_map = json.load(f)</code></p>"))
            display(HTML("<p><code>canonical = event_map.get(raw_label)  # None = ignore</code></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:{UI.ERROR_COLOR}'>{type(e).__name__}: {e}\n"
                         f"{traceback.format_exc()}</pre>"))


def run_verify(_=None):
    with out_verify:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            UI.log("⚠️ Run the scan (section 1) first.")
            return
        try:
            folder = STATE["folder"]
            mapping = load_existing_mapping()
            if not mapping:
                UI.log("⚠️ No mapping found yet: save it in section 4 first.")
                return
            items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))
            any_unmapped = False
            html = []
            for i, (labels, fids) in enumerate(items, 1):
                labs = sorted(labels)
                unmapped = [l for l in labs if l not in mapping]
                canon = sorted({mapping[l] for l in labs if mapping.get(l) is not None})
                if unmapped:
                    any_unmapped = True
                if verify_scope.value.startswith("Only") and not unmapped:
                    continue
                html.append(f"<h4>Configuration {i}: {len(fids)} file(s)</h4>")
                html.append("<b>Harmonized labels:</b> "
                            + (", ".join(f"<code>{c}</code>" for c in canon) or "<i>none</i>"))
                if unmapped:
                    html.append(f"<br><span style='color:{UI.ERROR_COLOR}'><b>Unmapped:</b> "
                                + ", ".join(f"<code>{u}</code>" for u in unmapped) + "</span>")
                html.append("<hr>")
            if not html:
                html.append("<i>Nothing to show for this scope.</i>")
            display(HTML("".join(html)))
            if any_unmapped:
                display(HTML(f"<p style='color:{UI.ERROR_COLOR}'><b>❌ Some raw labels are still unmapped.</b> "
                             "Map them in section 3 and save again.</p>"))
            else:
                display(HTML(f"<p style='color:{UI.OK_COLOR}'><b>✅ All raw labels are mapped "
                             "(or explicitly ignored).</b></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:{UI.ERROR_COLOR}'>{type(e).__name__}: {e}</pre>"))


def run_events_consistency_check(_=None):
    with out_check:
        clear_output()
        if not getattr(chooser, "selected_path", None):
            UI.log("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = SF.list_recordings(folder)
        if not edfs:
            UI.log("⚠️ No recording (.edf, .vhdr, .cdt) found.")
            return

        def _summary(counter):
            # per-canonical-type counts, e.g. "snore×426; hypopnea×2"
            return "; ".join(f"{lbl}×{n}" for lbl, n in counter.most_common())

        decl = declaration()     # the kinds unticked in 'Event files of this study' are not looked for
        if not decl['xml'] or not (decl['txt'] or decl['csv']):
            print("ℹ️ This check needs the XML and a text export or a CSV, ticked in 'Event files of this study'.")
            return
        rows = []
        for edf in edfs:
            fid = edf.stem
            txt, csv, xml = event_companion_paths(edf, decl['txt'], decl['csv'], decl['xml'])
            # Primary text source = the .txt if present, else the CSV; compared against the XML.
            primary = "txt" if txt is not None else ("csv" if csv is not None else "")
            blank = dict(file_id=fid, primary_source=primary, match_tol_s=tol_seconds.value,
                         n_primary="", n_xml="",
                         n_matched="", n_cooccur_difflabel="",
                         only_in_primary="", only_in_xml="", cooccur_label_pairs="")
            if primary == "" or xml is None:
                rows.append({**blank,
                             "status": "missing text/CSV" if primary == "" else "missing XML"})
                continue
            try:
                if primary == "txt":
                    rec_start = RL.read_start_datetime(edf)
                    if rec_start is None:
                        rows.append({**blank, "status": "recording start unknown (or the recording was paused)"})
                        continue
                    pev = _canon_events(load_events_from_txt(txt, rec_start))
                else:
                    pev = _canon_events(load_events_from_csv(csv))
                xev = _canon_events(load_events_from_xml(xml))
            except Exception as e:
                rows.append({**blank, "status": f"error: {type(e).__name__}"})
                continue

            # Match events across sources within +/- tol seconds (see _match_events).
            n_matched, cooccur, only_p, only_x = _match_events(pev, xev, tol_seconds.value)
            n_cooccur = sum(cooccur.values())
            n_only_p, n_only_x = sum(only_p.values()), sum(only_x.values())
            status = "match" if (n_only_p == 0 and n_only_x == 0 and n_cooccur == 0) else "MISMATCH"
            pair_str = "; ".join(f"{pl}↔{xl}×{n}" for (pl, xl), n in cooccur.most_common(10))
            rows.append(dict(
                file_id=fid, primary_source=primary, match_tol_s=tol_seconds.value,
                n_primary=len(pev), n_xml=len(xev),
                n_matched=n_matched, n_cooccur_difflabel=n_cooccur,
                only_in_primary=_summary(only_p), only_in_xml=_summary(only_x),
                cooccur_label_pairs=pair_str, status=status))
        df = pd.DataFrame(rows)
        cfgdir = folder / "config_param"
        cfgdir.mkdir(exist_ok=True)
        df.to_csv(cfgdir / "event_source_mismatch.tsv", sep="\t", index=False)
        n_match = int((df["status"] == "match").sum())
        n_mis = int((df["status"] == "MISMATCH").sum())
        n_other = len(df) - n_match - n_mis
        color = UI.OK_COLOR if (n_mis == 0 and n_other == 0) else UI.WARN_COLOR
        display(HTML(f"<p style='color:{color}'><b>{n_match} match</b>, {n_mis} mismatch, "
                     f"{n_other} missing-companion/error: of {len(df)} file(s).</p>"))
        display(HTML(f"<p style='color:{UI.MUTED_COLOR}'>Events matched by <b>canonical label + start time within "
                     f"±{tol_seconds.value}s</b>. <code>cooccur_label_pairs</code> lists events paired in "
                     "time but with <b>different labels</b>: likely the same event with an unharmonized "
                     "(e.g. cross-language) name; if the paired labels are unrelated they may instead be "
                     "two distinct events falling within the tolerance.</p>"))
        display(HTML(f"<p style='color:{UI.MUTED_COLOR}'>Saved <code>{cfgdir / 'event_source_mismatch.tsv'}</code></p>"))
        show = df[df["status"] != "match"]
        if len(show):
            display(HTML("<b>Files needing attention:</b>"))
            display(HTML(show.to_html(index=False)))


# ========================= Wiring & layout =========================
chooser.register_callback(_update_info)
for _cb in cb_files.values():
    _cb.observe(_show_suffix_boxes, names="value")
save_files_button.on_click(on_save_files)
fname_text.observe(_update_advanced_mark, names="value")
skip_existing.observe(lambda ch: render_harmonize(), names="value")
run_scan_button.on_click(run_scan)
run_check_button.on_click(run_events_consistency_check)
preview_save_button.on_click(on_preview_save)
validate_button.on_click(on_validate)
verify_button.on_click(run_verify)

ui_layout = widgets.VBox([
    section1, chooser, inputs_list, files_row, files_info, suffix_boxes['txt'], suffix_boxes['csv'],
    suffix_boxes['summary'], widgets.HBox([save_files_button, out_files]),
    existing_info, skip_existing, advanced, run_scan_button, scan_progress_row, out_scan,
    section1bis, widgets.HBox([run_check_button, tol_seconds]), out_check,
    section2, out_configs,
    section3, out_harmonize, validate_button, out_validate,
    section4, preview_save_button, out_save,
    section5, widgets.HBox([verify_scope, verify_button]), out_verify,
])
display(ui_layout)


<hr style="height:4px; background-color:black; border:none;">

## References

- Troester MM, Quan SF, Berry RB, et al. The AASM manual for the scoring of sleep and associated events: rules,
  terminology and technical specifications. Version 3. Darien (IL): American Academy of Sleep Medicine; 2023
  (the definitions of the arousals, respiratory events, limb movements and desaturations).